In [ ]:
from PIL import Image
from torch.utils.tensorboard import SummaryWriter

img_path = "E:/Users/15807/Documents/OneDrive - mail.dhu.edu.cn/学习资料/研究生资料/日常资料/DeepLearning/study/PYTORCH-TUTORIAL/data/test_producelabels/train/ants_image/0013035.jpg"

writer = SummaryWriter(log_dir="logs")

img = Image.open(img_path)

from torchvision import transforms

#ToTensor
trans_totensor = transforms.ToTensor()
img_tensor = trans_totensor(img)
writer.add_image("test", img_tensor)

#Normalize
print(img_tensor[0][0][0])
trans_norm = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
img_norm = trans_norm(img_tensor)
print(img_norm[0][0][0])
writer.add_image("test_norm", img_norm)

#Normalize2
print(img_tensor[0][0][0])
trans_norm2 = transforms.Normalize(mean=[1, 1, 1], std=[3, 2, 1])
img_norm2 = trans_norm2(img_tensor)
print(img_norm2[0][0][0])
writer.add_image("test_norm2", img_norm2)

#Normalize3
print(img_tensor[0][0][0])
trans_norm3 = transforms.Normalize(mean=[0.5, 0.3, 0.1], std=[1, 1, 2])
img_norm3 = trans_norm3(img_tensor)
print(img_norm3[0][0][0])
writer.add_image("test_norm3", img_norm3)


writer.close()

tensor(0.3137)
tensor(-0.3725)
tensor(0.3137)
tensor(-0.2288)
tensor(0.3137)
tensor(-0.1863)


In [ ]:
#使用tensorboard的命令是    python -m tensorboard.main --logdir logs --port 6006

为什么img_norm2的图片是黑色


因为你这句：

```python
trans_norm2 = transforms.Normalize(mean=[1, 1, 1], std=[3, 2, 1])
```

把 `ToTensor()` 后本来在 $[0, 1]$ 的像素又整体往下减了 1。

所以每个通道都会变成：

$$
x' = \frac{x - 1}{\text{std}}
$$

而 $x$ 本来最多也就 1，所以：

- 第一通道：$(x-1)/3$，范围大概是 $[-0.333, 0]$
- 第二通道：$(x-1)/2$，范围大概是 $[-0.5, 0]$
- 第三通道：$x-1$，范围是 $[-1, 0]$

也就是说，`img_norm2` 里的值几乎全是 0 或负数。

而 `writer.add_image(...)` 显示图像时，负数通常会被截断成 0，0 就是黑色，所以你看到的就是一张几乎全黑的图。

简单说就是：

- `Normalize` 不只是“美化一下数据”，它会真的改数值范围
- 你这里的 `mean=1` 太大了，把图像整体压到 0 以下
- TensorBoard 显示时把这些负值当成黑色处理了

如果你想看到正常的效果，可以试这两种：

```python
trans_norm2 = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
```

或者只把 `img_norm2` 打印出来看数值，不直接拿它当“可视化图片”。

如果你愿意，我可以顺手帮你解释一下为什么 `img_norm` 还能看起来正常，而 `img_norm2` 会黑得这么明显。